In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
emp_data = [(1,'manish',50000,'IT','m'),
(2,'vikash',60000,'sales','m'),
(3,'raushan',70000,'marketing','m'),
(4,'mukesh',80000,'IT','m'),
(5,'priti',90000,'sales','f'),
(6,'nikita',45000,'marketing','f'),
(7,'ragini',55000,'marketing','f'),
(8,'rashi',100000,'IT','f'),
(9,'aditya',65000,'IT','m'),
(10,'rahul',50000,'marketing','m'),
(11,'rakhi',50000,'IT','f'),
(12,'akhilesh',90000,'sales','m')]

emp_schema = ['id','name','salary','dept','gender']
emp_df = spark.createDataFrame(data=emp_data,schema=emp_schema)

## Group BY

In [0]:
emp_df.groupBy("dept")\
    .agg(sum("salary")).show()

+---------+-----------+
|     dept|sum(salary)|
+---------+-----------+
|marketing|     220000|
|    sales|     240000|
|       IT|     345000|
+---------+-----------+



#### But we need data for each dept wise, so we will use window function.
**We can acheive this by join after group by.** <br>
**We can acheive this using Partition.**<br>
**In below showing diff b/w row_number, rank, dense_rank.**

In [0]:
from pyspark.sql.window import Window
window=Window.partitionBy("dept").orderBy("salary")
emp_df.withColumn("row_number", row_number().over(window))\
    .withColumn("Rank",rank().over(window))\
    .withColumn("Dense_Rank",dense_rank().over(window))\
.show(truncate=False)

+---+--------+------+---------+------+----------+----+----------+
|id |name    |salary|dept     |gender|row_number|Rank|Dense_Rank|
+---+--------+------+---------+------+----------+----+----------+
|1  |manish  |50000 |IT       |m     |1         |1   |1         |
|11 |rakhi   |50000 |IT       |f     |2         |1   |1         |
|9  |aditya  |65000 |IT       |m     |3         |3   |2         |
|4  |mukesh  |80000 |IT       |m     |4         |4   |3         |
|8  |rashi   |100000|IT       |f     |5         |5   |4         |
|6  |nikita  |45000 |marketing|f     |1         |1   |1         |
|10 |rahul   |50000 |marketing|m     |2         |2   |2         |
|7  |ragini  |55000 |marketing|f     |3         |3   |3         |
|3  |raushan |70000 |marketing|m     |4         |4   |4         |
|2  |vikash  |60000 |sales    |m     |1         |1   |1         |
|5  |priti   |90000 |sales    |f     |2         |2   |2         |
|12 |akhilesh|90000 |sales    |m     |3         |2   |2         |
+---+-----

In [0]:
window=Window.partitionBy("dept").orderBy(desc("salary"))
emp_df.withColumn("row_number", row_number().over(window))\
    .withColumn("Rank",rank().over(window))\
    .withColumn("Dense_Rank",dense_rank().over(window))\
        .filter(col("Dense_Rank")<=2)\
.show(truncate=False)

+---+--------+------+---------+------+----------+----+----------+
|id |name    |salary|dept     |gender|row_number|Rank|Dense_Rank|
+---+--------+------+---------+------+----------+----+----------+
|8  |rashi   |100000|IT       |f     |1         |1   |1         |
|4  |mukesh  |80000 |IT       |m     |2         |2   |2         |
|3  |raushan |70000 |marketing|m     |1         |1   |1         |
|7  |ragini  |55000 |marketing|f     |2         |2   |2         |
|5  |priti   |90000 |sales    |f     |1         |1   |1         |
|12 |akhilesh|90000 |sales    |m     |2         |1   |1         |
|2  |vikash  |60000 |sales    |m     |3         |3   |2         |
+---+--------+------+---------+------+----------+----+----------+



In [0]:
window=Window.partitionBy("dept","gender").orderBy(desc("salary"))
emp_df.withColumn("row_number", row_number().over(window))\
    .withColumn("Rank",rank().over(window))\
    .withColumn("Dense_Rank",dense_rank().over(window))\
        .filter(col("Dense_Rank")<=2)\
.show(truncate=False)

+---+--------+------+---------+------+----------+----+----------+
|id |name    |salary|dept     |gender|row_number|Rank|Dense_Rank|
+---+--------+------+---------+------+----------+----+----------+
|8  |rashi   |100000|IT       |f     |1         |1   |1         |
|11 |rakhi   |50000 |IT       |f     |2         |2   |2         |
|4  |mukesh  |80000 |IT       |m     |1         |1   |1         |
|9  |aditya  |65000 |IT       |m     |2         |2   |2         |
|7  |ragini  |55000 |marketing|f     |1         |1   |1         |
|6  |nikita  |45000 |marketing|f     |2         |2   |2         |
|3  |raushan |70000 |marketing|m     |1         |1   |1         |
|10 |rahul   |50000 |marketing|m     |2         |2   |2         |
|5  |priti   |90000 |sales    |f     |1         |1   |1         |
|12 |akhilesh|90000 |sales    |m     |1         |1   |1         |
|2  |vikash  |60000 |sales    |m     |2         |2   |2         |
+---+--------+------+---------+------+----------+----+----------+

